In [1]:
import os
import tensorflow as tf

os.environ['CUDA_VISIBLE_DEVICES'] = '3'
gpus = tf.config.list_physical_devices('GPU')
tf.config.experimental.set_memory_growth(gpus[0], True)
tf.config.experimental.set_virtual_device_configuration(
    gpus[0],
    [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=1024)]
)

I0000 00:00:1775606820.637776    2974 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
import os
import torch
import base64
from tqdm import tqdm
from functools import lru_cache
from typing import List, Dict, Any, Iterator

from faster_whisper import WhisperModel
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
from huggingface_hub import InferenceClient

/data/koposovti/lecture-transcriber/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
HF_TOKEN = ...

In [ ]:
@lru_cache(maxsize=1)
def load_models(
    whisper_model_size: str = "medium",
    t5_model_path: str = "../models/rut5-cleaner-tuned/",
    e5_linker_path: str = "../models/lecture_linker_e5/",
    hf_token: str | None = None
) -> Dict[str, Any]:
    print("Loading models into memory...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"

    whisper_model = WhisperModel(whisper_model_size, device=device, compute_type=compute_type)

    t5_tokenizer = AutoTokenizer.from_pretrained(t5_model_path, token=hf_token)
    t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_model_path, token=hf_token).to(device)
    t5_model.eval()

    e5_linker = SentenceTransformer(e5_linker_path, token=hf_token, device=device)

    return {
        "whisper": whisper_model,
        "t5_tokenizer": t5_tokenizer,
        "t5_model": t5_model,
        "e5_linker": e5_linker
    }

In [ ]:
def transcribe_audio(audio_path: str, whisper_model: WhisperModel) -> Iterator[Dict[str, Any]]:
    print(f"Transcribing {audio_path}...")
    segments, _ = whisper_model.transcribe(audio_path, beam_size=5, language="ru", vad_filter=True)
    
    for segment in tqdm(segments):
        yield {
            "start": segment.start,
            "end": segment.end,
            "text": segment.text
        }

def group_segments(segments: Iterator[Dict], max_chars: int = 700) -> Iterator[str]:
    current_chunk = ""
    sentence_endings = ('.', '!', '?', '...')
    
    for seg in segments:
        text = seg['text'].strip()
        if not text: continue
        
        current_chunk += " " + text
        if len(current_chunk) >= max_chars and text.endswith(sentence_endings):
            yield current_chunk.strip()
            current_chunk = ""
            
    if current_chunk.strip():
        yield current_chunk.strip()

In [ ]:
class SequentialLinker:
    def __init__(self, model, materials: List[Dict], lookahead_window: int = 3):
        self.model = model
        self.materials = materials
        
        self.corpus_texts = [m["content"] for m in materials]
        self.corpus_ids = [m["id"] for m in materials]
        
        print("Encoding materials for semantic search...")
        formatted_passages = ["passage: " + text for text in self.corpus_texts]
        self.corpus_embeddings = self.model.encode(formatted_passages, convert_to_tensor=True)
        
        self.current_idx = 0
        self.lookahead_window = lookahead_window
        
    def predict(self, spoken_text: str):
        query_emb = self.model.encode("query: " + spoken_text, convert_to_tensor=True)
        cos_scores = util.cos_sim(query_emb, self.corpus_embeddings)[0]
        
        start_idx = self.current_idx
        end_idx = min(self.current_idx + self.lookahead_window + 1, len(cos_scores))
        
        window_scores = cos_scores[start_idx:end_idx]
        local_best_idx = int(torch.argmax(window_scores).item())
        best_score = window_scores[local_best_idx].item()
        
        global_best_idx = start_idx + local_best_idx
        
        if best_score > 0.40:
            self.current_idx = global_best_idx
            
        return {
            "matched_id": self.corpus_ids[global_best_idx],
            "matched_content": self.corpus_texts[global_best_idx],
            "score": round(best_score, 4)
        }

In [ ]:
def clean_text_chunks(chunks: Iterator[str], model, tokenizer) -> Iterator[str]:
    print(f"Cleaning text chunks...")
    device = model.device
    
    for text in chunks:
        input_text = "clean: " + text
        inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=512,
                num_beams=5,  
                repetition_penalty=2.5,
                length_penalty=1.0,
                early_stopping=True
            )
        yield tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def parse_pdf_to_slides_with_vlm(pdf_path: str, hf_token: str) -> List[Dict]:
    print(f"Parsing PDF and extracting visual descriptions for {pdf_path}...")
    slides = []
    doc = fitz.open(pdf_path)
    
    client = InferenceClient(model="Qwen/Qwen2-VL-7B-Instruct", token=hf_token)
    
    for i, page in enumerate(tqdm(doc)):
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        img_bytes = pix.tobytes("jpeg")
        base64_image = base64.b64encode(img_bytes).decode('utf-8')
        image_data_url = f"data:image/jpeg;base64,{base64_image}"
        
        # try:
        response = client.chat_completion(
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "Опиши содержание сайда во всех деталях. Объясни код, формулы, таблицы и схемыю Опиши что изображено на картинках."},
                        {"type": "image_url", "image_url": {"url": image_data_url}}
                    ]
                }
            ],
            max_tokens=500
        )
        vlm_description = response.choices[0].message.content
        # except Exception as e:
        #     print(f"HF API Error on slide {i+1}: {e}. Using basic text extraction insteaed.")
        #     vlm_description = page.get_text("text").strip()

        slides.append({
            "id": f"{os.path.basename(pdf_path)}_Slide_{i+1}", 
            "content": vlm_description
        })
            
    doc.close()
    return slides

In [ ]:
def parse_code_to_blocks(file_path: str) -> List[Dict]:
    """Parses source code files using Langchain language-aware text splitters."""
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    
    ext = file_path.split('.')[-1].lower()
    
    lang_map = {
        'py': Language.PYTHON, 'cpp': Language.CPP, 'c': Language.C,
        'h': Language.CPP, 'hpp': Language.CPP, 'cu': Language.CPP,
        'cuh': Language.CPP, 'java': Language.JAVA, 'go': Language.GO,
        'rs': Language.RUST,
    }
    
    lang = lang_map.get(ext)
    
    if lang:
        splitter = RecursiveCharacterTextSplitter.from_language(
            language=lang, chunk_size=1500, chunk_overlap=150
        )
    else:
        splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
        
    blocks = splitter.split_text(content)
        
    formatted_blocks = []
    for i, b in enumerate(blocks):
        if len(b.strip()) > 20:
            formatted_blocks.append({
                "id": f"{os.path.basename(file_path)}_block_{i+1}",
                "content": b.strip()
            })
            
    return formatted_blocks

In [16]:
def parse_materials(file_paths: List[str], hf_token: str) -> List[Dict]:
    all_materials = []
    for path in file_paths:
        if path.lower().endswith(".pdf"):
            all_materials.extend(parse_pdf_to_slides_with_vlm(path, hf_token))
        else:
            all_materials.extend(parse_code_to_blocks(path))
    return all_materials

In [ ]:
def link_speech_to_materials(
    speech_chunks: Iterator[str], 
    materials: List[Dict], 
    linker_model: SentenceTransformer
) -> Iterator[Dict[str, Any]]:
    if not materials:
        print("No materials provided to link against.")
        for chunk in speech_chunks:
            yield {"speech": chunk, "matched_material_id": None, "score": 0.0}
            return

    print("Linking speech to materials...")
    linker = SequentialLinker(linker_model, materials)
    
    for chunk in speech_chunks:
        match_info = linker.predict(chunk)
        
        yield {
            "speech_summary": chunk,
            "matched_material_id": match_info["matched_id"],
            "matched_material_content": match_info["matched_content"],
            "similarity_score": match_info["score"]
        }

In [ ]:
def process_lecture_files(file_list: List[str]) -> Iterator[Dict]:
    audio_files = [f for f in file_list if f.lower().endswith(('.mp3', '.wav', '.m4a'))]
    material_files = [f for f in file_list if not f.lower().endswith(('.mp3', '.wav', '.m4a', '.json', '.csv'))]
    
    if not audio_files:
        raise ValueError("No audio file found in the input list.")
    if len(audio_files) > 1:
        print("Multiple audio files found. Processing only the first one.")
        
    audio_path = audio_files[0]
    
    models = load_models()
    
    materials = parse_materials(material_files, HF_TOKEN)
    
    raw_segments = transcribe_audio(audio_path, models["whisper"])
    raw_chunks = group_segments(raw_segments)
    
    cleaned_speech = clean_text_chunks(raw_chunks, models["t5_model"], models["t5_tokenizer"])
    
    final_output = link_speech_to_materials(cleaned_speech, materials, models["e5_linker"])
    
    print("Pipeline finished successfully!")
    return final_output


In [ ]:
import glob

input_files = glob.glob("../data/raw/01/*")

# try:
results = process_lecture_files(input_files)

for idx, item in enumerate(results):
    print(f"\n--- Chunk {idx + 1} ---")
    print(f"Speech : {item['speech_summary']}")
    print(f"Slide/Code ID : {item['matched_material_id']} (Score: {item['similarity_score']})")
        
# except Exception as e:
#     print(f"Pipeline failed: {e}")

Parsing PDF and extracting visual descriptions for ../data/raw/01/cuda-02-sms.pdf...


  0%|          | 0/25 [00:00<?, ?it/s]


StopIteration: 

In [1]:
import torch
print(torch.__version__)

2.5.1+cu121
